# UTS Text Mining — Text Processing Bahasa Indonesia

**Dataset:** `dataset/dataset_dummy_ulasan.csv` (30 ulasan produk bhs. Indonesia + label sentimen)

**Alur:** Loading → Cleaning → Case Folding → Tokenizing → Stopword Removal → Stemming → TF-IDF → Visualisasi

> Cara pakai: Run All. Tidak butuh internet kecuali kalau mau install `Sastrawi` (sudah ada fallback).

## 1. Setup Library

In [ ]:
import re
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

# Stemming Sastrawi (opsional, ada fallback)
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    print("Sastrawi OK")
except ImportError:
    stemmer = None
    print("Sastrawi tidak terinstall → pakai fallback. Install via: pip install Sastrawi")

## 2. Load & Kenalan dengan Dataset

**Skema dataset (`dataset_dummy_ulasan.csv`):**

| kolom | arti | contoh |
|---|---|---|
| `id` | nomor unik | 1 |
| `ulasan` | teks mentah ulasan user | "Produk bagus banget..." |
| `rating` | 1–5 | 5 |
| `label` | sentimen manual | positif / netral / negatif |

* 30 baris, seimbang: 12 positif, 11 negatif, 7 netral.
* Sengaja berisik: huruf kapital, tanda seru, typo, singkatan — biar tahap cleaning terlihat gunanya.

In [ ]:
df = pd.read_csv("dataset/dataset_dummy_ulasan.csv")
print("Shape:", df.shape)
print(df["label"].value_counts())
print(df.head(5).to_string())

## 3. Cleaning + Case Folding

**Kenapa?** Teks mentah ada URL, mention, hashtag, angka, tanda baca, emoji. Model hanya butuh kata bermakna.

**Yang dilakukan `clean_text`:** lowercase → hapus URL/mention/hashtag → hapus angka → hapus tanda baca → rapikan spasi.

In [ ]:
def clean_text(teks: str) -> str:
    teks = teks.lower()                       # case folding
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)  # hapus URL
    teks = re.sub(r"@[A-Za-z0-9_]+", " ", teks)      # hapus mention
    teks = re.sub(r"#[A-Za-z0-9_]+", " ", teks)      # hapus hashtag
    teks = re.sub(r"[^a-z\s]", " ", teks)            # sisakan huruf a-z + spasi
    teks = re.sub(r"\s+", " ", teks).strip()        # rapikan spasi
    return teks

df["clean"] = df["ulasan"].astype(str).apply(clean_text)
print("SEBELUM:", df.loc[0, "ulasan"])
print("SESUDAH:", df.loc[0, "clean"])

## 4. Tokenizing

**Kenapa?** Kalimat → list kata (token) supaya bisa dihitung frekuensinya.

Pakai `split()` karena teks sudah bersih (hanya huruf + spasi). Setara `word_tokenize` untuk kasus sederhana.

In [ ]:
df["tokens"] = df["clean"].str.split()
print(df.loc[0, "tokens"])
print("Jumlah token kalimat-1:", len(df.loc[0, "tokens"]))

## 5. Stopword Removal

**Kenapa?** Kata seperti `yang, dan, di, tidak, sangat` muncul di mana-mana tapi tidak membedakan sentimen → dibuang agar fitur fokus ke kata bermakna (`bagus, kecewa, rusak`).

List manual 40 kata agar notebook jalan tanpa download NLTK.

In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan",
    "tidak", "tak", "bukan", "sangat", "sekali", "banget", "juga",
    "sudah", "telah", "ada", "adalah", "atau", "tapi", "namun",
    "karena", "jika", "kalau", "saat", "ketika", "saya", "kamu",
    "dia", "mereka", "kami", "kita", "nya", "lah", "kok", "sih",
    "aja", "saja", "pada", "dalam", "secara", "sering", "selalu"
}

df["no_stop"] = df["tokens"].apply(lambda toks: [t for t in toks if t not in STOPWORDS_ID])
print("SEBELUM:", df.loc[0, "tokens"])
print("SESUDAH:", df.loc[0, "no_stop"])

## 6. Stemming (Sastrawi)

**Kenapa?** `pengiriman, dikirim, mengirim` → akar kata `kirim` supaya dihitung sebagai 1 fitur, bukan 3 fitur berbeda.

Pakai Sastrawi (Nazief-Adriani untuk bhs. Indonesia). Kalau belum install, ada fallback penghapus imbuhan sederhana.

In [ ]:
def simple_stem_fallback(kata: str) -> str:
    """Fallback konservatif (aman, tidak merusak kata).
    Hanya hapus awalan/akhiran yang aman. Untuk imbuhan nasal
    (meng-/peng-/meny-/peny-) WAJIB pakai Sastrawi agar benar.
    Contoh: pengiriman -> kirim (Sastrawi), bukan irim (fallback kasar).
    """
    for pre in ("ber", "ter", "di", "ke", "se"):
        if kata.startswith(pre) and len(kata) > len(pre) + 3:
            kata = kata[len(pre):]
            break
    for suf in ("kan", "an", "nya", "lah", "kah", "i"):
        if kata.endswith(suf) and len(kata) > len(suf) + 3:
            kata = kata[:-len(suf)]
            break
    return kata

def stem_tokens(tokens):
    if stemmer is not None:
        return [stemmer.stem(t) for t in tokens]
    return [simple_stem_fallback(t) for t in tokens]

df["stemmed"] = df["no_stop"].apply(stem_tokens)
df["final"] = df["stemmed"].apply(lambda toks: " ".join(toks))
print("SEBELUM:", df.loc[0, "no_stop"])
print("SESUDAH:", df.loc[0, "stemmed"])
print("FINAL  :", df.loc[0, "final"])

## 7. TF-IDF + Visualisasi

**Kenapa TF-IDF?**

* `TF` = seberapa sering kata muncul di 1 dokumen.
* `IDF` = menghukum kata yang muncul di semua dokumen (mis. `produk`).
* Hasil: matriks angka siap masuk model ML (Naive Bayes / SVM / Logistic Regression).

Lalu plot 15 kata paling sering setelah preprocessing sebagai cek sanity.

In [ ]:
vectorizer = TfidfVectorizer(max_features=100)
X = vectorizer.fit_transform(df["final"])
print("Bentuk matriks TF-IDF:", X.shape)  # (30 dokumen, <=100 fitur)
print("Contoh fitur:", list(vectorizer.get_feature_names_out()[:10]))

# Visualisasi frekuensi
freq = Counter(w for toks in df["stemmed"] for w in toks)
top15 = freq.most_common(15)
kata, hitung = zip(*top15)

plt.figure()
plt.barh(list(kata)[::-1], list(hitung)[::-1])
plt.xlabel("Frekuensi")
plt.title("Top 15 Kata Setelah Preprocessing")
plt.tight_layout()
plt.show()

print("Top 15:", top15)

## 8. Simpan Hasil & Kesimpulan

* `dataset/ulasan_bersih.csv` = versi bersih siap training.
* Urutan WAJIB: cleaning → tokenizing → stopword → stemming → TF-IDF. Dibalik (mis. stemming sebelum cleaning) hasilnya rusak.
* Next step UTS: split train-test → klasifikasi sentimen → evaluasi accuracy/F1.

In [ ]:
df[["id", "ulasan", "clean", "final", "label"]].to_csv("dataset/ulasan_bersih.csv", index=False)
print("Tersimpan: dataset/ulasan_bersih.csv")
print(df[["ulasan", "final", "label"]].head(10).to_string())